# Entrega 2: tratamento e camada analítica

Lê os arquivos brutos gerados pela entrega 1, transforma cada fonte em painel tabular, calcula indicadores, constrói os rótulos de insolvência e entrega a camada analítica com a delimitação de **Minas Gerais**.

```
coleta (entrega 1)  ->  tratamento  ->  delimitacao MG  ->  camada analitica
```

| Camada | Unidade | Rótulo | Onde a UF entra |
|---|---|---|---|
| A. CVM | Companhia aberta e ano | Recuperação judicial ou falência | UF da sede, do cadastro CVM |
| B. BACEN | Instituição financeira e ano | Liquidação extrajudicial, intervenção ou RAET | UF da sede, do cadastro IF.data |
| C. Receita Federal | Empresa da coorte | Baixa cadastral na janela | UF do estabelecimento matriz |

## Uma decisão que precisa ficar explícita

Filtrar as três camadas por Minas Gerais tem efeitos muito diferentes em cada uma.

Na **camada C** o filtro é natural e obrigatório: é o que torna os 7,7 GB da Receita Federal processáveis, e sobram centenas de milhares de empresas. É onde a delimitação por MG faz sentido pleno.

Nas camadas **A e B** o filtro reduz drasticamente a amostra. Companhias abertas com sede em Minas são poucas dezenas, e eventos de recuperação judicial entre elas seriam pouquíssimos. O mesmo vale para instituições financeiras mineiras.

Por isso este notebook **gera as duas versões de cada camada**: a nacional e o recorte MG, com a coluna `uf` presente em todas. A camada C sai apenas em MG, por volume. Assim você decide na modelagem, e a delimitação vira uma dimensão de análise em vez de uma amputação da amostra. As contagens impressas na seção 6 mostram o custo exato de cada opção.

## 1. Configuração

In [1]:
!pip install -q pandas numpy pyarrow duckdb tqdm

In [3]:
import re
import io
import csv
import json
import shutil
import zipfile
import warnings
import unicodedata
from pathlib import Path
from datetime import datetime
from difflib import get_close_matches

import numpy as np
import pandas as pd
import duckdb
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
csv.field_size_limit(10 ** 7)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

`RAIZ` aponta para a pasta do trabalho, onde estão os brutos e onde ficam os resultados.

`DIR_TRABALHO` precisa ser uma pasta **local**, fora do Google Drive. A descompactação dos arquivos da Receita Federal escreve dezenas de GB temporários, e pastas sincronizadas falham com `Errno 22` nesse volume. Só os resultados finais voltam para a raiz.

In [6]:
RAIZ = Path(r"G:\Meu Drive\TCC\Damaris Rebeca")
DIR_TRABALHO = Path(r"C:\tcc_trabalho")

DIR_BRUTO = RAIZ / "01_bruto"
DIR_CVM = DIR_BRUTO / "cvm"
DIR_IFDATA = DIR_BRUTO / "bcb_ifdata"
DIR_RES = DIR_BRUTO / "bcb_resolucao"
DIR_RFB = DIR_BRUTO / "rfb"

DIR_INTER = RAIZ / "02_intermediario"
DIR_FINAL = RAIZ / "03_analitico"
DIR_RFB_CSV = DIR_TRABALHO / "rfb_csv"

for d in (DIR_INTER, DIR_FINAL, DIR_TRABALHO, DIR_RFB_CSV):
    d.mkdir(parents=True, exist_ok=True)

UF_RECORTE = "MG"
ANOS_COORTE = (2015, 2016, 2017, 2018, 2019)
JANELA_ANOS = 5
IFDATA_TIPOS = [1004, 1009]

alvo = str(DIR_TRABALHO).lower()
if any(x in alvo for x in ["onedrive", "google drive", "meu drive", "dropbox", "\\\\"]):
    print("ATENCAO: DIR_TRABALHO parece sincronizado com nuvem. Troque por uma pasta local.")

print("Brutos:", DIR_BRUTO, "| existe:", DIR_BRUTO.exists())
print("Trabalho:", DIR_TRABALHO)
print("Saida:", DIR_FINAL)
print("Recorte:", UF_RECORTE, "| coorte RFB:", ANOS_COORTE, "| janela:", JANELA_ANOS, "anos")

Brutos: G:\Meu Drive\TCC\Damaris Rebeca\01_bruto | existe: True
Trabalho: C:\tcc_trabalho
Saida: G:\Meu Drive\TCC\Damaris Rebeca\03_analitico
Recorte: MG | coorte RFB: (2015, 2016, 2017, 2018, 2019) | janela: 5 anos


In [8]:
def normalizar(s):
    s = unicodedata.normalize("NFKD", str(s) if pd.notna(s) else "")
    s = s.encode("ascii", "ignore").decode("ascii")
    return " ".join(s.upper().split())

def mb(n):
    return f"{n/1e6:,.1f} MB"

def div(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    return np.where((b == 0) | b.isna() | a.isna(), np.nan, a / b)

def salvar(df, nome, pasta=DIR_FINAL):
    caminho = Path(pasta) / f"{nome}.parquet"
    df.to_parquet(caminho, index=False)
    print(f"{nome}: {len(df):,} linhas x {df.shape[1]} colunas -> {caminho.name}")
    return caminho

def achar_coluna(df, *padroes):
    normalizadas = {c: normalizar(c) for c in df.columns}
    for p in padroes:
        alvo = normalizar(p)
        for c, n in normalizadas.items():
            if n == alvo:
                return c
    for p in padroes:
        alvo = normalizar(p)
        for c, n in normalizadas.items():
            if alvo and alvo in n:
                return c
    return None

---
# 2. Camada A: CVM, companhias abertas

## 2.1 Leitura dos demonstrativos

Regras aplicadas, todas documentáveis na metodologia:

1. Encoding `latin-1` e separador `;`, padrão dos arquivos da CVM.
2. Mantém apenas `ORDEM_EXERC` igual a `ÚLTIMO`, descartando a coluna comparativa que duplicaria observações.
3. Havendo retificações, mantém a maior `VERSAO`.
4. Normaliza a escala monetária: `ESCALA_MOEDA` igual a `MIL` multiplica por 1.000.
5. Usa o demonstrativo consolidado e cai para o individual quando a companhia não publica consolidado.

In [11]:
COLS_DF = ["CNPJ_CIA", "DENOM_CIA", "CD_CVM", "DT_REFER", "VERSAO", "DT_FIM_EXERC",
           "ORDEM_EXERC", "ESCALA_MOEDA", "CD_CONTA", "DS_CONTA", "VL_CONTA"]

def ler_demonstrativo(zip_path, prefixo, ano, tipo):
    nome = f"dfp_cia_aberta_{prefixo}_{tipo}_{ano}.csv"
    with zipfile.ZipFile(zip_path) as z:
        if nome not in z.namelist():
            return None
        with z.open(nome) as f:
            df = pd.read_csv(f, sep=";", encoding="latin-1", decimal=".", low_memory=False)
    if any(c not in df.columns for c in COLS_DF):
        return None
    df = df[COLS_DF].copy()
    df["ANO"] = ano
    df["TIPO_DEMO"] = tipo
    return df

zips = sorted(DIR_CVM.glob("dfp_*.zip"))
blocos = []
for zp in tqdm(zips, desc="CVM"):
    ano = int(re.findall(r"(\d{4})", zp.stem)[-1])
    for prefixo in ("BPA", "BPP", "DRE"):
        for tipo in ("con", "ind"):
            parte = ler_demonstrativo(zp, prefixo, ano, tipo)
            if parte is not None:
                blocos.append(parte)

bruto = pd.concat(blocos, ignore_index=True)
del blocos
print(f"{len(bruto):,} registros contabeis brutos")

CVM:   0%|          | 0/16 [00:00<?, ?it/s]

6,706,202 registros contabeis brutos


In [12]:
bruto = bruto[bruto["ORDEM_EXERC"].map(normalizar) == "ULTIMO"].copy()
bruto["VL_CONTA"] = pd.to_numeric(bruto["VL_CONTA"], errors="coerce")
fator = bruto["ESCALA_MOEDA"].str.upper().str.strip().map({"MIL": 1000.0, "UNIDADE": 1.0}).fillna(1.0)
bruto["VALOR"] = bruto["VL_CONTA"] * fator
bruto["DT_FIM_EXERC"] = pd.to_datetime(bruto["DT_FIM_EXERC"], errors="coerce")
bruto["CNPJ_CIA"] = bruto["CNPJ_CIA"].astype(str).str.replace(r"\D", "", regex=True).str.zfill(14)
bruto = bruto.sort_values("VERSAO").drop_duplicates(
    subset=["CNPJ_CIA", "ANO", "TIPO_DEMO", "CD_CONTA"], keep="last")

print(f"{len(bruto):,} registros apos limpeza | {bruto['CNPJ_CIA'].nunique():,} companhias")

3,346,441 registros apos limpeza | 1,181 companhias


## 2.2 Painel empresa-ano

O mapa traduz os códigos do plano de contas padronizado da CVM para nomes legíveis. São os códigos oficiais, então a banca consegue conferir conta a conta.

In [14]:
CONTAS = {
    "1": "ativo_total", "1.01": "ativo_circulante", "1.01.01": "caixa_equivalentes",
    "1.01.02": "aplicacoes_financeiras", "1.01.03": "contas_receber", "1.01.04": "estoques",
    "1.02": "ativo_nao_circulante", "1.02.01": "realizavel_longo_prazo",
    "2": "passivo_total", "2.01": "passivo_circulante", "2.01.04": "emprestimos_curto_prazo",
    "2.02": "passivo_nao_circulante", "2.02.01": "emprestimos_longo_prazo",
    "2.03": "patrimonio_liquido", "2.03.05": "lucros_acumulados",
    "3.01": "receita_liquida", "3.03": "resultado_bruto", "3.05": "ebit",
    "3.06": "resultado_financeiro", "3.06.01": "receitas_financeiras",
    "3.06.02": "despesas_financeiras", "3.07": "resultado_antes_tributos",
    "3.11": "lucro_liquido",
}

sel = bruto[bruto["CD_CONTA"].isin(CONTAS)].copy()
sel["CONTA"] = sel["CD_CONTA"].map(CONTAS)

painel = (sel.pivot_table(index=["CNPJ_CIA", "DENOM_CIA", "CD_CVM", "ANO", "TIPO_DEMO", "DT_FIM_EXERC"],
                          columns="CONTA", values="VALOR", aggfunc="first").reset_index())
painel.columns.name = None

con = painel[painel["TIPO_DEMO"] == "con"]
ind = painel[painel["TIPO_DEMO"] == "ind"]
chaves_con = set(zip(con["CNPJ_CIA"], con["ANO"]))
ind = ind[~pd.Series(list(zip(ind["CNPJ_CIA"], ind["ANO"])), index=ind.index).isin(chaves_con)]

painel = pd.concat([con, ind], ignore_index=True).sort_values(["CNPJ_CIA", "ANO"]).reset_index(drop=True)
print(f"{len(painel):,} observacoes empresa-ano | {painel['CNPJ_CIA'].nunique():,} companhias")
print(painel.groupby("TIPO_DEMO").size().to_dict())

10,172 observacoes empresa-ano | 1,181 companhias
{'con': 5946, 'ind': 4226}


## 2.3 Indicadores e escores clássicos

Vinte e sete indicadores em cinco blocos, mais o Termômetro de Kanitz (1978) e o Z'' de Altman para mercados emergentes. Os dois escores entram como preditores e, principalmente, como linha de base: permitem responder se o machine learning supera os modelos clássicos de insolvência, e em quanto.

In [16]:
d = painel
d["exigivel_total"] = d["passivo_circulante"].fillna(0) + d["passivo_nao_circulante"].fillna(0)
d["divida_bruta"] = d["emprestimos_curto_prazo"].fillna(0) + d["emprestimos_longo_prazo"].fillna(0)
d["capital_circulante_liquido"] = d["ativo_circulante"] - d["passivo_circulante"]
d["disponibilidades"] = d["caixa_equivalentes"].fillna(0) + d["aplicacoes_financeiras"].fillna(0)

d["liquidez_corrente"] = div(d["ativo_circulante"], d["passivo_circulante"])
d["liquidez_seca"] = div(d["ativo_circulante"] - d["estoques"].fillna(0), d["passivo_circulante"])
d["liquidez_imediata"] = div(d["disponibilidades"], d["passivo_circulante"])
d["liquidez_geral"] = div(d["ativo_circulante"].fillna(0) + d["realizavel_longo_prazo"].fillna(0), d["exigivel_total"])

d["endividamento_geral"] = div(d["exigivel_total"], d["ativo_total"])
d["endividamento_financeiro"] = div(d["divida_bruta"], d["ativo_total"])
d["composicao_endividamento"] = div(d["passivo_circulante"], d["exigivel_total"])
d["alavancagem"] = div(d["exigivel_total"], d["patrimonio_liquido"])
d["pl_sobre_ativo"] = div(d["patrimonio_liquido"], d["ativo_total"])
d["pl_sobre_exigivel"] = div(d["patrimonio_liquido"], d["exigivel_total"])
d["imobilizacao_recursos"] = div(d["ativo_nao_circulante"], d["patrimonio_liquido"])
d["divida_liquida_ativo"] = div(d["divida_bruta"] - d["disponibilidades"], d["ativo_total"])

d["roa"] = div(d["lucro_liquido"], d["ativo_total"])
d["roe"] = div(d["lucro_liquido"], d["patrimonio_liquido"])
d["margem_liquida"] = div(d["lucro_liquido"], d["receita_liquida"])
d["margem_bruta"] = div(d["resultado_bruto"], d["receita_liquida"])
d["margem_operacional"] = div(d["ebit"], d["receita_liquida"])
d["ebit_sobre_ativo"] = div(d["ebit"], d["ativo_total"])
d["lucros_acumulados_ativo"] = div(d["lucros_acumulados"], d["ativo_total"])
d["retorno_antes_tributos"] = div(d["resultado_antes_tributos"], d["ativo_total"])

d["giro_ativo"] = div(d["receita_liquida"], d["ativo_total"])
d["ccl_sobre_ativo"] = div(d["capital_circulante_liquido"], d["ativo_total"])
d["capital_giro_receita"] = div(d["capital_circulante_liquido"], d["receita_liquida"])
d["cobertura_juros"] = div(d["ebit"], d["despesas_financeiras"].abs())
d["despesa_financeira_receita"] = div(d["despesas_financeiras"].abs(), d["receita_liquida"])

d = d.sort_values(["CNPJ_CIA", "ANO"])
d["crescimento_receita"] = d.groupby("CNPJ_CIA")["receita_liquida"].pct_change()
d["crescimento_ativo"] = d.groupby("CNPJ_CIA")["ativo_total"].pct_change()
d["pl_negativo"] = (d["patrimonio_liquido"] < 0).astype("int8")
d["prejuizo"] = (d["lucro_liquido"] < 0).astype("int8")

k1 = div(d["lucro_liquido"], d["patrimonio_liquido"])
d["kanitz_fi"] = (0.05 * k1 + 1.65 * d["liquidez_geral"] + 3.55 * d["liquidez_seca"]
                  - 1.06 * d["liquidez_corrente"] - 0.33 * div(d["exigivel_total"], d["patrimonio_liquido"]))
d["kanitz_faixa"] = pd.cut(d["kanitz_fi"], [-np.inf, -3, 0, np.inf],
                           labels=["insolvente", "penumbra", "solvente"])
d["altman_z2_emergentes"] = (3.25 + 6.56 * d["ccl_sobre_ativo"] + 3.26 * d["lucros_acumulados_ativo"]
                             + 6.72 * d["ebit_sobre_ativo"] + 1.05 * d["pl_sobre_exigivel"])

painel = d.reset_index(drop=True)
print(f"{painel.shape[1]} colunas no painel")
print(painel["kanitz_faixa"].value_counts(dropna=False).to_dict())

65 colunas no painel
{'solvente': 8641, nan: 658, 'penumbra': 488, 'insolvente': 385}


## 2.4 Rótulo de insolvência

A própria CVM carimba a situação na razão social e no campo `SIT_EMISSOR`, o que transforma a curadoria manual em extração rastreável. Três cuidados que a versão anterior não tinha:

**A expressão é casada por inteiro.** `RECUPERAÇÃO JUDICIAL` e `RECUPERAÇÃO EXTRAJUDICIAL`, nunca apenas `RECUPERA`, que capturava falsos positivos como "Cia de Recuperação Secundária", uma empresa de extração de petróleo.

**A data só é aceita quando é confiável.** Quando o marcador vem de `SIT_EMISSOR`, a data é `DT_INI_SIT_EMISSOR`. Quando vem do motivo de cancelamento, é `DT_CANCEL`. Quando vem apenas da razão social, **a data é desconhecida** e o caso vai para um arquivo de pesquisa manual em vez de receber um chute. Foi esse chute que datava a recuperação judicial da InterCement em 2004 em vez de 2024.

**Recuperação extrajudicial é separada da judicial.** São institutos distintos na Lei 11.101, e misturar os dois é erro conceitual.

Instituições financeiras também são sinalizadas, porque liquidação de banco é o evento da camada B e ter as mesmas entidades nas duas camadas contamina a comparação.

In [18]:
cad = pd.read_csv(DIR_CVM / "cad_cia_aberta.csv", sep=";", encoding="latin-1", dtype=str)
cad["CNPJ_CIA"] = cad["CNPJ_CIA"].str.replace(r"\D", "", regex=True).str.zfill(14)
for c in ["DENOM_SOCIAL", "SIT", "SIT_EMISSOR", "MOTIVO_CANCEL", "SETOR_ATIV",
          "DT_INI_SIT_EMISSOR", "DT_CANCEL", "UF"]:
    if c not in cad.columns:
        cad[c] = None

n_denom = cad["DENOM_SOCIAL"].map(normalizar)
n_emis = cad["SIT_EMISSOR"].map(normalizar)
n_motiv = cad["MOTIVO_CANCEL"].map(normalizar)
n_setor = cad["SETOR_ATIV"].map(normalizar)

P_RJ, P_REX, P_FAL = r"RECUPERACAO JUDICIAL", r"RECUPERACAO EXTRAJUDICIAL", r"FALENCIA|LIQUIDACAO EXTRAJUDICIAL"

marca_rj = n_denom.str.contains(P_RJ, regex=True) | n_emis.str.contains(P_RJ, regex=True)
marca_rex = n_denom.str.contains(P_REX, regex=True) | n_emis.str.contains(P_REX, regex=True)
marca_fal = (n_denom.str.contains(P_FAL, regex=True) | n_emis.str.contains(P_FAL, regex=True)
             | n_motiv.str.contains(P_FAL, regex=True))
tem_marca = marca_rj | marca_rex | marca_fal
emis_marca = n_emis.str.contains(f"{P_RJ}|{P_REX}|{P_FAL}", regex=True)

dt_emis = pd.to_datetime(cad["DT_INI_SIT_EMISSOR"], errors="coerce")
dt_canc = pd.to_datetime(cad["DT_CANCEL"], errors="coerce")
data_evento = pd.Series(pd.NaT, index=cad.index)
origem = pd.Series("a pesquisar", index=cad.index)

usa_emis = tem_marca & emis_marca & dt_emis.notna()
data_evento[usa_emis] = dt_emis[usa_emis]
origem[usa_emis] = "DT_INI_SIT_EMISSOR"
usa_canc = tem_marca & data_evento.isna() & n_motiv.str.contains(P_FAL, regex=True) & dt_canc.notna()
data_evento[usa_canc] = dt_canc[usa_canc]
origem[usa_canc] = "DT_CANCEL"

eventos = pd.DataFrame({
    "cnpj": cad["CNPJ_CIA"], "denominacao": cad["DENOM_SOCIAL"],
    "tipo_evento": np.where(marca_rj, "recuperacao_judicial",
                     np.where(marca_rex, "recuperacao_extrajudicial",
                     np.where(marca_fal, "falencia_liquidacao", ""))),
    "data_evento": data_evento, "ano_evento": data_evento.dt.year,
    "origem_data": origem, "setor": cad["SETOR_ATIV"],
    "financeira": n_setor.str.contains(r"BANCO|ARRENDAMENTO|SEGURO|INTERMEDIACAO|CREDITO|FINANCEIR|SECURITIZ", regex=True),
    "tambem_falencia": marca_fal, "sit": cad["SIT"], "sit_emissor": cad["SIT_EMISSOR"],
})[tem_marca].copy()

anos_painel = (int(painel["ANO"].min()), int(painel["ANO"].max()))
eventos["no_painel"] = eventos["cnpj"].isin(set(painel["CNPJ_CIA"]))
eventos["na_janela"] = eventos["ano_evento"].between(anos_painel[0] + 1, anos_painel[1] + 1)
eventos["utilizavel"] = eventos["no_painel"] & eventos["na_janela"] & (~eventos["financeira"])

print(f"{len(eventos)} candidatos | painel cobre {anos_painel[0]} a {anos_painel[1]}\n")
print(eventos.groupby(["tipo_evento", "origem_data"]).size().to_string())
print("\nutilizaveis por tipo:")
print(eventos[eventos["utilizavel"]].groupby("tipo_evento").size().to_string())

eventos.to_csv(DIR_INTER / "eventos_cvm.csv", index=False, encoding="utf-8-sig")
sem_data = eventos[(eventos["origem_data"] == "a pesquisar") & eventos["no_painel"]]
sem_data.to_csv(DIR_INTER / "eventos_cvm_a_pesquisar.csv", index=False, encoding="utf-8-sig")
print(f"\n{len(sem_data)} casos sem data confiavel gravados em eventos_cvm_a_pesquisar.csv")

69 candidatos | painel cobre 2011 a 2026

tipo_evento                origem_data       
falencia_liquidacao        DT_CANCEL              2
                           DT_INI_SIT_EMISSOR    16
                           a pesquisar            3
recuperacao_extrajudicial  DT_INI_SIT_EMISSOR     1
recuperacao_judicial       DT_INI_SIT_EMISSOR    42
                           a pesquisar            5

utilizaveis por tipo:
tipo_evento
falencia_liquidacao          14
recuperacao_extrajudicial     1
recuperacao_judicial         36

5 casos sem data confiavel gravados em eventos_cvm_a_pesquisar.csv


O arquivo `eventos_cvm_a_pesquisar.csv` lista as companhias que têm o carimbo na razão social mas cuja data do evento não pode ser derivada com segurança. Preencha `ano_evento` nele consultando o fato relevante na CVM ou a consulta processual, salve, e rode a célula seguinte com `USAR_PESQUISA_MANUAL = True` para incorporá-las.

In [20]:
USAR_PESQUISA_MANUAL = False

confirmados = eventos[eventos["utilizavel"] & eventos["ano_evento"].notna()][
    ["cnpj", "denominacao", "ano_evento", "tipo_evento", "origem_data"]].copy()

if USAR_PESQUISA_MANUAL:
    manual = pd.read_csv(DIR_INTER / "eventos_cvm_a_pesquisar.csv", dtype=str)
    manual["ano_evento"] = pd.to_numeric(manual["ano_evento"], errors="coerce")
    manual = manual[manual["ano_evento"].notna()]
    manual["origem_data"] = "pesquisa manual"
    confirmados = pd.concat([confirmados, manual[confirmados.columns]], ignore_index=True)

confirmados = confirmados.dropna(subset=["ano_evento"]).drop_duplicates(subset=["cnpj"], keep="first")
confirmados["ano_evento"] = confirmados["ano_evento"].astype(int)

EVENTOS_ALVO = ["recuperacao_judicial", "falencia_liquidacao"]
usados = confirmados[confirmados["tipo_evento"].isin(EVENTOS_ALVO)]

painel["y_rj"] = 0
painel["y_rj_h2"] = 0
painel["y_rj_h3"] = 0
painel["descartar_pos_evento"] = 0

for _, ev in usados.iterrows():
    m = painel["CNPJ_CIA"] == ev["cnpj"]
    ano = ev["ano_evento"]
    painel.loc[m & (painel["ANO"] == ano - 1), "y_rj"] = 1
    painel.loc[m & painel["ANO"].between(ano - 2, ano - 1), "y_rj_h2"] = 1
    painel.loc[m & painel["ANO"].between(ano - 3, ano - 1), "y_rj_h3"] = 1
    painel.loc[m & (painel["ANO"] >= ano), "descartar_pos_evento"] = 1

print(f"{len(usados)} eventos aplicados")
print(painel[["y_rj", "y_rj_h2", "y_rj_h3", "descartar_pos_evento"]].sum().to_dict())
print("taxa y_rj: {:.3%}".format(painel["y_rj"].mean()))
print("\neventos por ano:")
print(usados.groupby("ano_evento").size().to_string())

48 eventos aplicados
{'y_rj': 48, 'y_rj_h2': 93, 'y_rj_h3': 136, 'descartar_pos_evento': 158}
taxa y_rj: 0.472%

eventos por ano:
ano_evento
2012    3
2013    1
2014    4
2015    6
2016    3
2017    2
2018    4
2019    3
2020    3
2022    3
2023    4
2024    6
2025    3
2026    3


### Rótulo secundário: dificuldade financeira severa

Definição operacional: patrimônio líquido negativo, **ou** cobertura de juros abaixo de 1 acompanhada de prejuízo por dois exercícios consecutivos. A exigência simultânea de prejuízo é o que separa dificuldade real de simples custo financeiro alto, comum no Brasil com juros elevados. Serve como teste de robustez do rótulo primário e multiplica os casos positivos.

In [22]:
sinal = ((painel["cobertura_juros"] < 1) & (painel["lucro_liquido"] < 0)).astype("int8")
painel["sinal_distress"] = sinal
consec = (painel.groupby("CNPJ_CIA")["sinal_distress"]
                .rolling(2, min_periods=2).sum().reset_index(level=0, drop=True) == 2)
painel["y_distress"] = ((painel["pl_negativo"] == 1) | consec.fillna(False)).astype("int8")

print(painel["y_distress"].value_counts().to_dict())
print("taxa y_distress: {:.2%}".format(painel["y_distress"].mean()))

{0: 7786, 1: 2386}
taxa y_distress: 23.46%


## 2.5 Amostra pareada e gravação

Em vez de SMOTE sobre desbalanceamento extremo, a camada A usa amostra pareada por ano e porte, que é o desenho de Beaver (1966) e Altman (1968) e é muito mais fácil de defender. Cada caso positivo recebe três controles do mesmo ano e do ativo total mais próximo. O painel completo continua salvo, então os dois desenhos podem ser comparados na modelagem.

In [24]:
def parear(df, alvo, k=3):
    base = df[(df["descartar_pos_evento"] == 0) & df["ativo_total"].notna() & (df["ativo_total"] > 0)].copy()
    if base.empty or base[alvo].sum() == 0:
        return base.head(0)
    base["log_ativo"] = np.log(base["ativo_total"])
    partes = []
    for ano, grupo in base.groupby("ANO"):
        positivos = grupo[grupo[alvo] == 1]
        negativos = grupo[grupo[alvo] == 0]
        if positivos.empty or negativos.empty:
            continue
        usados = set()
        for _, p in positivos.iterrows():
            cand = negativos[~negativos.index.isin(usados)]
            if cand.empty:
                break
            escolhidos = (cand["log_ativo"] - p["log_ativo"]).abs().nsmallest(k).index
            usados.update(escolhidos)
            partes.append(cand.loc[escolhidos])
        partes.append(positivos)
    return pd.concat(partes).drop_duplicates().sort_values(["ANO", "CNPJ_CIA"]) if partes else base.head(0)

cadastro_meta = cad[["CNPJ_CIA", "SIT", "SETOR_ATIV", "UF"]].drop_duplicates(subset=["CNPJ_CIA"])
cadastro_meta = cadastro_meta.rename(columns={"SETOR_ATIV": "setor", "UF": "uf", "SIT": "situacao"})

camada_a = painel.merge(cadastro_meta, on="CNPJ_CIA", how="left")
camada_a["uf"] = camada_a["uf"].fillna("").str.strip().str.upper()
camada_a["fonte"] = "CVM"

pareado_rj = parear(camada_a, "y_rj")
pareado_distress = parear(camada_a, "y_distress")

salvar(camada_a, "camada_a_cvm_painel")
salvar(pareado_rj, "camada_a_cvm_pareado_rj")
salvar(pareado_distress, "camada_a_cvm_pareado_distress")

print("\ndistribuicao por UF (top 10):")
print(camada_a["uf"].value_counts().head(10).to_string())

camada_a_cvm_painel: 10,172 linhas x 75 colunas -> camada_a_cvm_painel.parquet
camada_a_cvm_pareado_rj: 192 linhas x 76 colunas -> camada_a_cvm_pareado_rj.parquet
camada_a_cvm_pareado_distress: 8,718 linhas x 76 colunas -> camada_a_cvm_pareado_distress.parquet

distribuicao por UF (top 10):
uf
SP    5014
RJ    1597
MG     773
RS     528
SC     470
PR     398
BA     227
CE     212
DF     135
       123


---
# 3. Camada B: BACEN, IF.data

## 3.1 Formato dos arquivos

O IF.data não entrega tabelas, entrega quatro peças que precisam ser montadas:

| Arquivo | Conteúdo |
|---|---|
| `cadastro{data}_{tipo}.json` | Uma linha por instituição. `c0` é o código, `c2` o nome, `c10` a UF, `c11` a cidade, `c12` o segmento da Resolução 4.553 |
| `info{data}.json` | Dicionário de campos: traduz o id numérico para o nome do indicador |
| `trel{data}_{id}.json` | Definição de colunas de cada relatório, indicando quais ids de campo pertencem a ele |
| `dados{data}_{n}.json` | Os valores, no formato `{"e": codigo, "v": [{"i": campo, "v": valor}]}` |

O `e` de cada linha de dados é o **código da instituição**, o mesmo `c0` do cadastro. Os arquivos `dados` não são separados por relatório, então todos são lidos e concatenados. Validei o casamento entre `e` e `c0`: bate 100% no tipo 1009.

In [26]:
RELATORIOS_ALVO = {normalizar(x) for x in
                   ["Resumo", "Ativo", "Passivo", "Demonstracao de Resultado",
                    "Informacoes de Capital", "Segmentacao"]}

COLS_CADASTRO = {"c0": "cod_inst", "c2": "instituicao", "c3": "tcb", "c5": "consolidacao",
                 "c7": "controle", "c10": "uf", "c11": "cidade", "c12": "segmento"}

def campos_do_relatorio(trel):
    ifds = set()
    pilha = list(trel.get("c", []) or [])
    while pilha:
        col = pilha.pop()
        if col.get("ifd") is not None:
            ifds.add(int(col["ifd"]))
        pilha.extend(col.get("sc", []) or [])
    return ifds

def ler_ifdata(dt):
    pasta = DIR_IFDATA / str(dt)
    if not pasta.exists():
        return None
    info_path = pasta / f"info{dt}.json"
    if not info_path.exists():
        return None
    info = json.loads(info_path.read_text(encoding="utf-8"))
    nome_campo = {int(x["id"]): str(x.get("n", "")).strip() for x in info}

    cadastro = tipo_usado = None
    for t in IFDATA_TIPOS:
        p = pasta / f"cadastro{dt}_{t}.json"
        if p.exists():
            cadastro = pd.DataFrame(json.loads(p.read_text(encoding="utf-8")))
            tipo_usado = t
            break
    if cadastro is None:
        return None

    ifds = set()
    for p in pasta.glob(f"trel{dt}_*.json"):
        try:
            t = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        t = t.get("trel", t)
        if normalizar(t.get("n", "")) not in RELATORIOS_ALVO:
            continue
        if not any(s.get("id") == tipo_usado for s in t.get("s", []) or []):
            continue
        ifds |= campos_do_relatorio(t)
    if not ifds:
        return None

    registros = []
    for p in sorted(pasta.glob(f"dados{dt}_*.json")):
        try:
            d = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        for linha in d.get("values", []) or []:
            e = linha.get("e")
            for c in linha.get("v", []) or []:
                i = c.get("i")
                if i in ifds:
                    registros.append((e, i, c.get("v")))
    if not registros:
        return None

    longo = pd.DataFrame(registros, columns=["cod_inst", "campo_id", "valor"])
    longo["cod_inst"] = pd.to_numeric(longo["cod_inst"], errors="coerce")
    longo["valor"] = pd.to_numeric(longo["valor"], errors="coerce")
    longo["campo"] = longo["campo_id"].map(nome_campo).fillna(longo["campo_id"].astype(str))
    largo = longo.pivot_table(index="cod_inst", columns="campo", values="valor", aggfunc="first").reset_index()
    largo.columns.name = None

    cols = {k: v for k, v in COLS_CADASTRO.items() if k in cadastro.columns}
    meta = cadastro[list(cols)].rename(columns=cols).copy()
    meta["cod_inst"] = pd.to_numeric(meta["cod_inst"], errors="coerce")
    meta = meta.dropna(subset=["cod_inst"]).drop_duplicates(subset=["cod_inst"])

    saida = meta.merge(largo, on="cod_inst", how="inner")
    saida.insert(0, "ano", int(str(dt)[:4]))
    saida.insert(1, "data_base", int(dt))
    saida.insert(2, "tipo_instituicao", tipo_usado)
    return saida

datas_disponiveis = sorted(int(p.name) for p in DIR_IFDATA.iterdir()
                           if p.is_dir() and p.name.isdigit())
print("datas-base em disco:", datas_disponiveis)

datas-base em disco: [201412, 201512, 201612, 201712, 201812, 201912, 202012, 202112, 202212, 202312, 202412, 202512]


In [27]:
partes = []
for dt in tqdm(datas_disponiveis, desc="IF.data"):
    parte = ler_ifdata(dt)
    if parte is None:
        print(f"{dt}: sem dados utilizaveis")
        continue
    partes.append(parte)
    print(f"{dt}: {len(parte):>5} instituicoes | {parte.shape[1]:>3} colunas | tipo {parte['tipo_instituicao'].iloc[0]}")

camada_b = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
print(f"\n{len(camada_b):,} observacoes instituicao-ano | {camada_b['cod_inst'].nunique():,} instituicoes")
print("anos:", sorted(camada_b["ano"].unique()))

IF.data:   0%|          | 0/12 [00:00<?, ?it/s]

201412:  1559 instituicoes |  79 colunas | tipo 1004
201512:  1499 instituicoes |  98 colunas | tipo 1004
201612:  1444 instituicoes |  98 colunas | tipo 1004
201712:  1393 instituicoes |  98 colunas | tipo 1004
201812:  1349 instituicoes |  98 colunas | tipo 1004
201912:  1301 instituicoes |  95 colunas | tipo 1004
202012:  1305 instituicoes |  94 colunas | tipo 1004
202112:  1309 instituicoes |  94 colunas | tipo 1004
202212:  1341 instituicoes |  94 colunas | tipo 1004
202312:  1349 instituicoes |  91 colunas | tipo 1009
202412:  1362 instituicoes |  89 colunas | tipo 1009
202512:  1341 instituicoes |  27 colunas | tipo 1009

16,552 observacoes instituicao-ano | 1,967 instituicoes
anos: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 3.2 Indicadores

Os nomes das colunas vêm do dicionário do próprio BACEN e mudam de redação ao longo dos anos. Por isso os indicadores são localizados por busca de nome, e a célula imprime exatamente qual coluna foi usada em cada caso. Confira essa lista antes de seguir.

In [29]:
MAPA_INDICADORES = {
    "ativo_total": ["Ativo Total"],
    "patrimonio_liquido": ["Patrimônio Líquido", "Patrimonio Liquido"],
    "lucro_liquido": ["Lucro Líquido", "Lucro Liquido", "Resultado Líquido"],
    "captacoes": ["Captações", "Captacoes"],
    "carteira_credito": ["Carteira de Crédito Classificada", "Operações de Crédito"],
    "passivo_circulante_elp": ["Passivo Circulante e Exigível a Longo Prazo"],
    "basileia": ["Índice de Basileia", "Indice de Basileia"],
    "capital_principal": ["Índice de Capital Principal"],
    "patrimonio_referencia": ["Patrimônio de Referência para Comparação com o RWA", "Patrimônio de Referência"],
    "rwa": ["Ativos Ponderados pelo Risco (RWA)", "Ativos Ponderados pelo Risco"],
    "provisao_credito": ["Provisão sobre Carteira de Crédito", "Provisão"],
}

encontrados = {}
for destino, padroes in MAPA_INDICADORES.items():
    col = achar_coluna(camada_b, *padroes)
    encontrados[destino] = col
    if col:
        camada_b[destino] = pd.to_numeric(camada_b[col], errors="coerce")

print(f"{'indicador':<26}{'coluna encontrada no IF.data'}")
for k, v in encontrados.items():
    print(f"{k:<26}{v if v else 'NAO ENCONTRADO'}")

indicador                 coluna encontrada no IF.data
ativo_total               Ativo Total
patrimonio_liquido        Patrimônio Líquido
lucro_liquido             Lucro Líquido 
(j) = (g) + (h) + (i)
captacoes                 Captações
carteira_credito          Carteira de Crédito Classificada
passivo_circulante_elp    Passivo Circulante e Exigível a Longo Prazo 
(h) = (e) + (f) + (g)
basileia                  Índice de Basileia 
(m) = (e) / (i)
capital_principal         Índice de Capital Principal 
(k) = (a) / (i)
patrimonio_referencia     Patrimônio de Referência para Comparação com o RWA
rwa                       Ativos Ponderados pelo Risco (RWA) 
(i) = (f) + (g) + (h)
provisao_credito          Arrendamento Mercantil Líquido de Provisão 
(e)


In [30]:
if "ativo_total" in camada_b.columns:
    camada_b["roa"] = div(camada_b.get("lucro_liquido"), camada_b["ativo_total"])
    camada_b["roe"] = div(camada_b.get("lucro_liquido"), camada_b.get("patrimonio_liquido"))
    camada_b["pl_sobre_ativo"] = div(camada_b.get("patrimonio_liquido"), camada_b["ativo_total"])
    camada_b["alavancagem"] = div(camada_b["ativo_total"], camada_b.get("patrimonio_liquido"))
    camada_b["credito_sobre_ativo"] = div(camada_b.get("carteira_credito"), camada_b["ativo_total"])
    camada_b["captacoes_sobre_ativo"] = div(camada_b.get("captacoes"), camada_b["ativo_total"])
    camada_b["prejuizo"] = (pd.to_numeric(camada_b.get("lucro_liquido"), errors="coerce") < 0).astype("int8")

camada_b = camada_b.sort_values(["cod_inst", "ano"])
camada_b["crescimento_ativo"] = camada_b.groupby("cod_inst")["ativo_total"].pct_change()
camada_b["uf"] = camada_b["uf"].fillna("").str.strip().str.upper()
camada_b["fonte"] = "BACEN"

indic = [c for c in ["roa", "roe", "pl_sobre_ativo", "alavancagem", "basileia",
                     "credito_sobre_ativo", "captacoes_sobre_ativo"] if c in camada_b.columns]
print(camada_b[indic].describe().T.to_string())
print("\ninstituicoes por UF (top 10):")
print(camada_b.drop_duplicates(subset=["cod_inst"])["uf"].value_counts().head(10).to_string())

                            count      mean         std             min    25%    50%    75%         max
roa                    7,242.0000    0.0134      0.1170         -4.3317 0.0054 0.0153 0.0301      1.7217
roe                    7,242.0000   -0.4056     36.7887     -3,129.8638 0.0217 0.0588 0.0936     14.7397
pl_sobre_ativo        15,207.0000 -160.5027 19,838.8788 -2,446,464.0000 0.1408 0.2253 0.6961      1.0000
alavancagem           15,205.0000    5.7911     61.8385       -135.0861 1.4195 4.4023 7.0573  7,574.3677
basileia               9,375.0000    2.2794    166.0541         -3.6036 0.1906 0.2865 0.6379 16,069.7317
credito_sobre_ativo   15,207.0000    0.4927      0.6683          0.0000 0.2849 0.5539 0.7023     72.5390
captacoes_sobre_ativo 15,207.0000    0.4095      0.3294          0.0000 0.0000 0.4869 0.7157      4.7473

instituicoes por UF (top 10):
uf
SP    687
MG    249
RS    185
PR    173
SC    146
RJ    137
BA     51
GO     50
ES     40
MT     29


## 3.3 Rótulo de resolução

A relação oficial do BACEN identifica as instituições por CNPJ e nome, enquanto o IF.data usa um código próprio. Não existe chave comum, então o pareamento é feito por **nome normalizado**, com similaridade. Como o número de casos é da ordem de algumas dezenas em uma década, a conferência manual é viável e necessária.

A célula grava `pareamento_resolucao.csv` com as sugestões e o grau de similaridade. Confira, corrija o que estiver errado, e rode a célula seguinte para aplicar.

In [32]:
arqs_res = sorted(p for p in DIR_RES.glob("*") if p.suffix.lower() in (".csv", ".xlsx", ".xls"))
partes_res = []
for p in arqs_res:
    try:
        if p.suffix.lower() == ".csv":
            try:
                t = pd.read_csv(p, sep=";", encoding="latin-1", dtype=str)
            except Exception:
                t = pd.read_csv(p, sep=",", encoding="utf-8", dtype=str)
        else:
            t = pd.read_excel(p, dtype=str)
        t["recurso"] = p.name
        partes_res.append(t)
        print(f"{p.name:<50} {len(t):>6} linhas | {list(t.columns)[:6]}")
    except Exception as e:
        print(f"{p.name}: ERRO {type(e).__name__}")

resolucao = pd.concat(partes_res, ignore_index=True) if partes_res else pd.DataFrame()
print(f"\n{len(resolucao)} registros de resolucao")


0 registros de resolucao


In [33]:
COL_NOME_RES = achar_coluna(resolucao, "instituicao", "nome", "razao social", "entidade") if len(resolucao) else None
COL_DATA_RES = achar_coluna(resolucao, "data", "decretacao", "inicio") if len(resolucao) else None
COL_TIPO_RES = achar_coluna(resolucao, "regime", "tipo", "situacao") if len(resolucao) else None

print("coluna de nome:", COL_NOME_RES)
print("coluna de data:", COL_DATA_RES)
print("coluna de regime:", COL_TIPO_RES)

pareamento = pd.DataFrame()
if COL_NOME_RES and COL_DATA_RES and len(camada_b):
    ev = resolucao[[c for c in [COL_NOME_RES, COL_DATA_RES, COL_TIPO_RES] if c]].copy()
    ev.columns = ["instituicao_resolucao", "data_evento"] + (["regime"] if COL_TIPO_RES else [])
    ev["data_evento"] = pd.to_datetime(ev["data_evento"], errors="coerce", dayfirst=True)
    ev = ev.dropna(subset=["data_evento"])
    ev["ano_evento"] = ev["data_evento"].dt.year
    ev["chave"] = ev["instituicao_resolucao"].map(normalizar)

    catalogo_if = camada_b.drop_duplicates(subset=["cod_inst"])[["cod_inst", "instituicao", "uf"]].copy()
    catalogo_if["chave"] = catalogo_if["instituicao"].map(normalizar)
    mapa_chave = dict(zip(catalogo_if["chave"], catalogo_if["cod_inst"]))
    chaves = list(mapa_chave)

    linhas = []
    for _, e in ev.iterrows():
        aprox = get_close_matches(e["chave"], chaves, n=1, cutoff=0.72)
        linhas.append({
            "instituicao_resolucao": e["instituicao_resolucao"],
            "ano_evento": int(e["ano_evento"]),
            "regime": e.get("regime", ""),
            "sugestao_ifdata": aprox[0] if aprox else "",
            "cod_inst": mapa_chave.get(aprox[0]) if aprox else None,
            "confirmado": "S" if aprox else "",
        })
    pareamento = pd.DataFrame(linhas)
    pareamento.to_csv(DIR_INTER / "pareamento_resolucao.csv", index=False, encoding="utf-8-sig")
    achou = pareamento["cod_inst"].notna().sum()
    print(f"\n{len(pareamento)} eventos | {achou} com correspondencia automatica")
    print(pareamento.head(20).to_string(index=False))

coluna de nome: None
coluna de data: None
coluna de regime: None


In [34]:
camada_b["y_resolucao"] = 0
camada_b["descartar_pos_evento"] = 0

if len(pareamento):
    validos = pareamento[(pareamento["confirmado"].astype(str).str.upper() == "S")
                         & pareamento["cod_inst"].notna()]
    for _, ev in validos.iterrows():
        m = camada_b["cod_inst"] == ev["cod_inst"]
        ano = int(ev["ano_evento"])
        camada_b.loc[m & (camada_b["ano"] == ano - 1), "y_resolucao"] = 1
        camada_b.loc[m & (camada_b["ano"] >= ano), "descartar_pos_evento"] = 1
    print(f"{len(validos)} eventos aplicados")

print(camada_b["y_resolucao"].value_counts().to_dict())
print("taxa: {:.3%}".format(camada_b["y_resolucao"].mean()))
salvar(camada_b, "camada_b_bacen_painel")

{0: 16552}
taxa: 0.000%
camada_b_bacen_painel: 16,552 linhas x 128 colunas -> camada_b_bacen_painel.parquet


WindowsPath('G:/Meu Drive/TCC/Damaris Rebeca/03_analitico/camada_b_bacen_painel.parquet')

---
# 4. Camada C: Receita Federal, coorte de Minas Gerais

## 4.1 Extração com filtro

Descompactar tudo daria mais de 30 GB. Como o recorte é Minas Gerais, o filtro é aplicado **durante a leitura do ZIP**, linha a linha, e só o que interessa vai para o disco. Isso derruba o volume para algo em torno de 300 MB e elimina de vez o `Errno 22`.

São duas passagens, porque só a tabela de estabelecimentos tem UF:

1. Varre `Estabelecimentos`, mantém as matrizes de MG abertas nos anos da coorte, e coleta o conjunto de `cnpj_basico`.
2. Varre `Empresas`, `Socios` e `Simples`, mantendo apenas as linhas cujo `cnpj_basico` está nesse conjunto.

As tabelas de domínio (CNAE, motivos, municípios, naturezas) são pequenas e saem inteiras.

In [36]:
FILTRAR_ANOS_ABERTURA = True

IDX_ESTAB_CNPJ, IDX_ESTAB_INICIO, IDX_ESTAB_UF = 0, 10, 19
anos_alvo = {str(a) for a in ANOS_COORTE}

def nome_seguro(texto):
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^A-Za-z0-9._-]", "_", texto)

def escrever_membro(zp, membro, manter, coletar_idx=None, coletados=None):
    destino = DIR_RFB_CSV / nome_seguro(f"{zp.stem}__{Path(membro).name}")
    if destino.exists() and destino.stat().st_size > 0:
        return 0, 0
    parcial = destino.with_name(destino.name + ".parcial")
    lidas = gravadas = 0
    with zipfile.ZipFile(zp) as z, z.open(membro) as bruto_zip:
        entrada = io.TextIOWrapper(bruto_zip, encoding="latin-1", newline="")
        leitor = csv.reader(entrada, delimiter=";", quotechar='"')
        with open(parcial, "w", encoding="latin-1", newline="", errors="replace") as saida:
            escritor = csv.writer(saida, delimiter=";", quotechar='"', quoting=csv.QUOTE_MINIMAL)
            lote = []
            for linha in leitor:
                lidas += 1
                if manter is not None and not manter(linha):
                    continue
                if coletar_idx is not None and coletados is not None and len(linha) > coletar_idx:
                    try:
                        coletados.add(int(linha[coletar_idx]))
                    except (TypeError, ValueError):
                        pass
                lote.append(linha)
                gravadas += 1
                if len(lote) >= 50000:
                    escritor.writerows(lote)
                    lote = []
            if lote:
                escritor.writerows(lote)
    parcial.replace(destino)
    return lidas, gravadas

def processar(padrao, manter, coletar_idx=None, coletados=None, rotulo=""):
    zips = sorted(DIR_RFB.glob(padrao))
    tl = tg = 0
    for zp in tqdm(zips, desc=rotulo or padrao):
        marcador = DIR_RFB_CSV / (nome_seguro(zp.stem) + ".ok")
        if marcador.exists():
            continue
        try:
            with zipfile.ZipFile(zp) as z:
                membros = [m for m in z.namelist() if not m.endswith("/")]
            for m in membros:
                lidas, gravadas = escrever_membro(zp, m, manter, coletar_idx, coletados)
                tl += lidas
                tg += gravadas
            marcador.touch()
        except Exception as e:
            print(f"{zp.name}: {type(e).__name__}: {str(e)[:80]}")
    taxa = tg / tl if tl else 0
    print(f"{rotulo}: {tl:,} lidas, {tg:,} gravadas ({taxa:.2%})")

def manter_estabelecimento(linha):
    if len(linha) <= IDX_ESTAB_UF:
        return False
    if linha[IDX_ESTAB_UF].strip().upper() != UF_RECORTE.upper():
        return False
    if FILTRAR_ANOS_ABERTURA and linha[IDX_ESTAB_INICIO][:4] not in anos_alvo:
        return False
    return True

print("UF:", UF_RECORTE, "| anos de abertura:", sorted(anos_alvo) if FILTRAR_ANOS_ABERTURA else "todos")
print("destino:", DIR_RFB_CSV)

UF: MG | anos de abertura: ['2015', '2016', '2017', '2018', '2019']
destino: C:\tcc_trabalho\rfb_csv


In [37]:
cnpjs_alvo = set()
processar("Estabelecimentos*.zip", manter_estabelecimento, IDX_ESTAB_CNPJ, cnpjs_alvo, "Estabelecimentos")
print(f"\n{len(cnpjs_alvo):,} CNPJ basicos selecionados")

def manter_por_cnpj(linha):
    if not linha:
        return False
    try:
        return int(linha[0]) in cnpjs_alvo
    except (TypeError, ValueError):
        return False

filtro = manter_por_cnpj if cnpjs_alvo else None
processar("Empresas*.zip", filtro, rotulo="Empresas")
processar("Socios*.zip", filtro, rotulo="Socios")
processar("Simples.zip", filtro, rotulo="Simples")
for dominio in ("Cnaes", "Motivos", "Municipios", "Naturezas", "Qualificacoes", "Paises"):
    processar(f"{dominio}.zip", None, rotulo=dominio)

extraidos = [p for p in DIR_RFB_CSV.rglob("*") if p.is_file() and p.suffix.lower() != ".ok"]
print(f"\n{len(extraidos)} arquivos | {mb(sum(p.stat().st_size for p in extraidos))}")

Estabelecimentos:   0%|          | 0/10 [00:00<?, ?it/s]

Estabelecimentos: 72,789,638 lidas, 1,453,462 gravadas (2.00%)

1,433,361 CNPJ basicos selecionados


Empresas:   0%|          | 0/10 [00:00<?, ?it/s]

Empresas: 69,523,304 lidas, 1,433,361 gravadas (2.06%)


Socios:   0%|          | 0/10 [00:00<?, ?it/s]

...

Naturezas: 91 lidas, 91 gravadas (100.00%)


Qualificacoes:   0%|          | 0/1 [00:00<?, ?it/s]

Qualificacoes: 68 lidas, 68 gravadas (100.00%)


Paises:   0%|          | 0/1 [00:00<?, ?it/s]

Paises: 255 lidas, 255 gravadas (100.00%)

37 arquivos | 451.9 MB


## 4.2 Carga em DuckDB e coorte

Os arquivos da Receita não têm cabeçalho, usam `;` como separador e encoding `latin-1`. O layout é oficial e está fixado por posição. DuckDB lê direto do disco, sem carregar tudo em memória, e resolve as junções em segundos.

In [39]:
CAMINHO_DB = DIR_TRABALHO / "rfb.duckdb"
if CAMINHO_DB.exists():
    CAMINHO_DB.unlink()
con_db = duckdb.connect(str(CAMINHO_DB))

COLS_EMPRESAS = ["cnpj_basico", "razao_social", "natureza_juridica", "qualificacao_responsavel",
                 "capital_social", "porte", "ente_federativo"]
COLS_ESTAB = ["cnpj_basico", "cnpj_ordem", "cnpj_dv", "identificador_matriz_filial", "nome_fantasia",
              "situacao_cadastral", "data_situacao_cadastral", "motivo_situacao_cadastral",
              "nome_cidade_exterior", "pais", "data_inicio_atividade", "cnae_fiscal_principal",
              "cnae_fiscal_secundaria", "tipo_logradouro", "logradouro", "numero", "complemento",
              "bairro", "cep", "uf", "municipio", "ddd_1", "telefone_1", "ddd_2", "telefone_2",
              "ddd_fax", "fax", "correio_eletronico", "situacao_especial", "data_situacao_especial"]
COLS_SOCIOS = ["cnpj_basico", "identificador_socio", "nome_socio", "cnpj_cpf_socio",
               "qualificacao_socio", "data_entrada_sociedade", "pais", "representante_legal",
               "nome_representante", "qualificacao_representante", "faixa_etaria"]
COLS_SIMPLES = ["cnpj_basico", "opcao_simples", "data_opcao_simples", "data_exclusao_simples",
                "opcao_mei", "data_opcao_mei", "data_exclusao_mei"]
COLS_DOMINIO = ["codigo", "descricao"]

def arquivos_por_marca(marcas):
    marcas = [m.upper() for m in marcas]
    return sorted(str(p) for p in DIR_RFB_CSV.rglob("*")
                  if p.is_file() and p.suffix.lower() != ".ok"
                  and any(m in p.name.upper() for m in marcas))

def registrar(nome, marcas, colunas):
    arquivos = arquivos_por_marca(marcas)
    if not arquivos:
        print(f"{nome}: nenhum arquivo casou com {marcas}")
        return
    lista = ", ".join("'" + a.replace("'", "''") + "'" for a in arquivos)
    nomes = ", ".join(f"'{c}'" for c in colunas)
    sql = (f"CREATE OR REPLACE VIEW {nome} AS SELECT * FROM read_csv([{lista}], "
           f"delim=';', header=false, quote='\"', escape='\"', encoding='latin-1', "
           f"all_varchar=true, names=[{nomes}], ignore_errors=true)")
    try:
        con_db.execute(sql)
    except Exception:
        con_db.execute(sql.replace("encoding='latin-1', ", ""))
    total = con_db.execute(f"SELECT COUNT(*) FROM {nome}").fetchone()[0]
    print(f"{nome:<20}{len(arquivos):>3} arquivos  {total:>12,} linhas")

registrar("empresas", ["EMPRECSV"], COLS_EMPRESAS)
registrar("estabelecimentos", ["ESTABELE"], COLS_ESTAB)
registrar("socios", ["SOCIOCSV"], COLS_SOCIOS)
registrar("simples", ["SIMPLES"], COLS_SIMPLES)
registrar("tab_cnae", ["CNAECSV"], COLS_DOMINIO)
registrar("tab_motivos", ["MOTICSV"], COLS_DOMINIO)
registrar("tab_municipios", ["MUNICCSV"], COLS_DOMINIO)
registrar("tab_naturezas", ["NATJUCSV"], COLS_DOMINIO)

empresas             10 arquivos     1,433,361 linhas
estabelecimentos     10 arquivos     1,453,462 linhas
socios               10 arquivos       410,823 linhas
simples               1 arquivos     1,274,755 linhas
tab_cnae              1 arquivos         1,359 linhas
tab_motivos           1 arquivos            63 linhas
tab_municipios        1 arquivos         5,572 linhas
tab_naturezas         1 arquivos            91 linhas


In [40]:
anos_sql = ", ".join(str(a) for a in ANOS_COORTE)

con_db.execute(f"""
CREATE OR REPLACE TABLE coorte AS
WITH est AS (
    SELECT e.cnpj_basico, e.uf, e.municipio, e.situacao_cadastral, e.motivo_situacao_cadastral,
           TRY_STRPTIME(e.data_inicio_atividade, '%Y%m%d') AS dt_abertura,
           TRY_STRPTIME(e.data_situacao_cadastral, '%Y%m%d') AS dt_situacao,
           e.cnae_fiscal_principal,
           LEFT(e.cnae_fiscal_principal, 2) AS cnae_divisao,
           LENGTH(COALESCE(e.cnae_fiscal_secundaria, '')) > 0 AS tem_cnae_secundario,
           LENGTH(COALESCE(e.correio_eletronico, '')) > 0 AS tem_email,
           LENGTH(COALESCE(e.telefone_1, '')) > 0 AS tem_telefone
    FROM estabelecimentos e
    WHERE e.identificador_matriz_filial = '1' AND e.uf = '{UF_RECORTE}'
),
soc AS (
    SELECT cnpj_basico, COUNT(*) AS qtd_socios,
           SUM(CASE WHEN identificador_socio = '2' THEN 1 ELSE 0 END) AS qtd_socios_pf,
           MIN(TRY_CAST(faixa_etaria AS INTEGER)) AS faixa_etaria_min,
           MAX(TRY_CAST(faixa_etaria AS INTEGER)) AS faixa_etaria_max
    FROM socios GROUP BY cnpj_basico
)
SELECT est.cnpj_basico, emp.razao_social, est.uf,
       mun.descricao AS municipio_nome, est.cnae_fiscal_principal, est.cnae_divisao,
       cna.descricao AS cnae_descricao, nat.descricao AS natureza_descricao,
       emp.natureza_juridica,
       TRY_CAST(REPLACE(emp.capital_social, ',', '.') AS DOUBLE) AS capital_social,
       emp.porte, est.dt_abertura, YEAR(est.dt_abertura) AS ano_abertura,
       MONTH(est.dt_abertura) AS mes_abertura, est.situacao_cadastral,
       est.motivo_situacao_cadastral, mot.descricao AS motivo_descricao, est.dt_situacao,
       est.tem_email, est.tem_telefone, est.tem_cnae_secundario,
       COALESCE(soc.qtd_socios, 0) AS qtd_socios,
       COALESCE(soc.qtd_socios_pf, 0) AS qtd_socios_pf,
       soc.faixa_etaria_min, soc.faixa_etaria_max,
       sim.opcao_simples, sim.opcao_mei,
       TRY_STRPTIME(sim.data_opcao_simples, '%Y%m%d') AS dt_opcao_simples,
       TRY_STRPTIME(sim.data_exclusao_simples, '%Y%m%d') AS dt_exclusao_simples
FROM est
LEFT JOIN empresas emp ON emp.cnpj_basico = est.cnpj_basico
LEFT JOIN soc ON soc.cnpj_basico = est.cnpj_basico
LEFT JOIN simples sim ON sim.cnpj_basico = est.cnpj_basico
LEFT JOIN tab_cnae cna ON cna.codigo = est.cnae_fiscal_principal
LEFT JOIN tab_motivos mot ON mot.codigo = est.motivo_situacao_cadastral
LEFT JOIN tab_municipios mun ON mun.codigo = est.municipio
LEFT JOIN tab_naturezas nat ON nat.codigo = emp.natureza_juridica
WHERE YEAR(est.dt_abertura) IN ({anos_sql})
""")

print(f"{con_db.execute('SELECT COUNT(*) FROM coorte').fetchone()[0]:,} empresas na coorte")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1,415,279 empresas na coorte


In [41]:
camada_c = con_db.execute("SELECT * FROM coorte").df()

camada_c["anos_ate_situacao"] = (camada_c["dt_situacao"] - camada_c["dt_abertura"]).dt.days / 365.25
camada_c["baixada"] = (camada_c["situacao_cadastral"] == "08").astype("int8")
dentro = camada_c["anos_ate_situacao"].between(0, JANELA_ANOS)
camada_c["y_baixa"] = (camada_c["baixada"].eq(1) & dentro).astype("int8")

camada_c["idade_empresa_anos"] = (pd.Timestamp.today() - camada_c["dt_abertura"]).dt.days / 365.25
camada_c["log_capital_social"] = np.log1p(camada_c["capital_social"].clip(lower=0))
camada_c["optante_simples"] = (camada_c["opcao_simples"] == "S").astype("int8")
camada_c["optante_mei"] = (camada_c["opcao_mei"] == "S").astype("int8")
camada_c["excluido_simples"] = camada_c["dt_exclusao_simples"].notna().astype("int8")
camada_c["porte_num"] = pd.to_numeric(camada_c["porte"], errors="coerce")
camada_c["fonte"] = "RFB"

print(f"{len(camada_c):,} empresas | taxa de baixa em {JANELA_ANOS} anos: {camada_c['y_baixa'].mean():.2%}\n")
print(camada_c.groupby("ano_abertura")["y_baixa"].agg(["count", "mean"]).to_string())
print("\nmotivos de baixa mais frequentes:")
print(camada_c[camada_c["baixada"] == 1]["motivo_descricao"].value_counts().head(8).to_string())

salvar(camada_c, "camada_c_rfb_coorte_mg")
con_db.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1,415,279 empresas | taxa de baixa em 5 anos: 37.82%

               count   mean
ano_abertura               
2015          218569 0.3301
2016          306148 0.5033
2017          251987 0.3443
2018          294626 0.3464
2019          343949 0.3495

motivos de baixa mais frequentes:
motivo_descricao
EXTINCAO POR ENCERRAMENTO LIQUIDACAO VOLUNTARIA      705545
OBITO DO MEI - TITULAR FALECIDO                       13399
REGISTRO CANCELADO                                     1484
INCORPORACAO                                            444
ENCERRAMENTO DA FALENCIA                                112
INEXISTENCIA DE FATO                                     30
EXTINCAO PELO ENCERRAMENTO DA LIQUIDACAO JUDICIAL        29
ENCERRAMENTO DA LIQUIDACAO                               16
camada_c_rfb_coorte_mg: 1,415,279 linhas x 39 colunas -> camada_c_rfb_coorte_mg.parquet


---
# 5. Delimitação por Minas Gerais

A camada C já nasce restrita a MG. As camadas A e B são recortadas aqui, e as duas versões ficam gravadas: a nacional e a mineira. A tabela impressa mostra o custo exato do recorte em cada uma, principalmente em número de casos positivos, que é o que limita a modelagem.

In [43]:
def recortar(df, nome_base, coluna_uf="uf"):
    if df is None or df.empty:
        print(f"{nome_base}: vazio")
        return None
    mg = df[df[coluna_uf].fillna("").str.upper().str.strip() == UF_RECORTE].copy()
    salvar(df, f"{nome_base}_nacional")
    salvar(mg, f"{nome_base}_{UF_RECORTE.lower()}")
    return mg

camada_a_mg = recortar(camada_a, "camada_a_cvm")
camada_b_mg = recortar(camada_b, "camada_b_bacen")

camada_a_cvm_nacional: 10,172 linhas x 75 colunas -> camada_a_cvm_nacional.parquet
camada_a_cvm_mg: 773 linhas x 75 colunas -> camada_a_cvm_mg.parquet
camada_b_bacen_nacional: 16,552 linhas x 128 colunas -> camada_b_bacen_nacional.parquet
camada_b_bacen_mg: 2,478 linhas x 128 colunas -> camada_b_bacen_mg.parquet


In [44]:
def perfil(nome, df, alvo, id_col):
    if df is None or df.empty or alvo not in df.columns:
        return {"camada": nome, "observacoes": 0, "unidades": 0, "positivos": 0, "taxa": np.nan}
    pos = int(df[alvo].sum())
    return {"camada": nome, "observacoes": len(df), "unidades": df[id_col].nunique(),
            "positivos": pos, "taxa": pos / len(df) if len(df) else np.nan}

linhas = [
    perfil("A CVM nacional, y_rj", camada_a, "y_rj", "CNPJ_CIA"),
    perfil(f"A CVM {UF_RECORTE}, y_rj", camada_a_mg, "y_rj", "CNPJ_CIA"),
    perfil("A CVM nacional, y_distress", camada_a, "y_distress", "CNPJ_CIA"),
    perfil(f"A CVM {UF_RECORTE}, y_distress", camada_a_mg, "y_distress", "CNPJ_CIA"),
    perfil("B BACEN nacional", camada_b, "y_resolucao", "cod_inst"),
    perfil(f"B BACEN {UF_RECORTE}", camada_b_mg, "y_resolucao", "cod_inst"),
    perfil(f"C RFB {UF_RECORTE}", camada_c, "y_baixa", "cnpj_basico"),
]

comparativo = pd.DataFrame(linhas)
comparativo["taxa"] = comparativo["taxa"].map(lambda v: f"{v:.2%}" if pd.notna(v) else "")
comparativo.to_csv(DIR_FINAL / "comparativo_camadas.csv", index=False, encoding="utf-8-sig")
print(comparativo.to_string(index=False))

                    camada  observacoes  unidades  positivos   taxa
      A CVM nacional, y_rj        10172      1181         48  0.47%
            A CVM MG, y_rj          773        86          3  0.39%
A CVM nacional, y_distress        10172      1181       2386 23.46%
      A CVM MG, y_distress          773        86        132 17.08%
          B BACEN nacional        16552      1967          0  0.00%
                B BACEN MG         2478       254          0  0.00%
                  C RFB MG      1415279   1415279     535270 37.82%


### Como ler essa tabela

Se a linha de MG na camada A vier com menos de 30 positivos, o modelo daquela camada **não deve ser estimado só com Minas**. O caminho nesse caso é treinar no nacional e usar a coluna `uf` como variável, ou apresentar Minas como um recorte descritivo dentro do modelo nacional. Isso é uma constatação empírica sobre o tamanho da amostra, não uma escolha de gosto, e escrita assim na metodologia fica sólida.

A camada C é onde a delimitação mineira sustenta a modelagem por conta própria, com centenas de milhares de empresas.

---
# 6. Verificação final

Checa o que costuma derrubar um trabalho na banca: rótulo vazio, chave duplicada, indicador em escala absurda e vazamento temporal.

In [47]:
def verificar(nome, df, chave, alvo, indicadores):
    if df is None or df.empty:
        return {"base": nome, "linhas": 0, "problemas": ["base vazia"]}
    problemas = []
    if alvo not in df.columns:
        problemas.append("rotulo ausente")
    else:
        pos = int(df[alvo].sum())
        if pos == 0:
            problemas.append("nenhum caso positivo")
        elif pos < 30:
            problemas.append(f"apenas {pos} positivos, insuficiente para K-Fold estratificado com k=10")
    faltam = [c for c in chave if c not in df.columns]
    if faltam:
        problemas.append(f"chave incompleta, faltam {faltam}")
    else:
        dup = int(df.duplicated(subset=chave).sum())
        if dup:
            problemas.append(f"{dup} linhas duplicadas na chave {chave}")
    for c in indicadores:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        if s.notna().sum() == 0:
            problemas.append(f"{c} totalmente vazio")
        elif s.isna().mean() > 0.5:
            problemas.append(f"{c} com {s.isna().mean():.0%} de ausentes")
        elif s.abs().max() > 1e6:
            problemas.append(f"{c} com valor extremo ({s.abs().max():,.0f}), revisar winsorizacao")
    return {"base": nome, "linhas": len(df), "problemas": problemas or ["ok"]}

ind_a = ["liquidez_corrente", "endividamento_geral", "roa", "roe", "giro_ativo",
         "cobertura_juros", "kanitz_fi", "altman_z2_emergentes"]
ind_b = [c for c in ["roa", "roe", "pl_sobre_ativo", "basileia", "alavancagem"] if c in (camada_b.columns if len(camada_b) else [])]
ind_c = ["log_capital_social", "qtd_socios", "idade_empresa_anos"]

for d in [
    verificar("A CVM nacional, y_rj", camada_a, ["CNPJ_CIA", "ANO"], "y_rj", ind_a),
    verificar("A CVM nacional, y_distress", camada_a, ["CNPJ_CIA", "ANO"], "y_distress", ind_a),
    verificar(f"A CVM {UF_RECORTE}", camada_a_mg, ["CNPJ_CIA", "ANO"], "y_rj", ind_a),
    verificar("B BACEN nacional", camada_b, ["cod_inst", "ano"], "y_resolucao", ind_b),
    verificar(f"B BACEN {UF_RECORTE}", camada_b_mg, ["cod_inst", "ano"], "y_resolucao", ind_b),
    verificar(f"C RFB {UF_RECORTE}", camada_c, ["cnpj_basico"], "y_baixa", ind_c),
]:
    print(f"\n{d['base']} ({d['linhas']:,} linhas)")
    for p in d["problemas"]:
        print("   -", p)


A CVM nacional, y_rj (10,172 linhas)
   - 1 linhas duplicadas na chave ['CNPJ_CIA', 'ANO']
   - giro_ativo com valor extremo (4,010,405), revisar winsorizacao
   - kanitz_fi com valor extremo (2,563,220), revisar winsorizacao
   - altman_z2_emergentes com valor extremo (2,402,848), revisar winsorizacao

A CVM nacional, y_distress (10,172 linhas)
   - 1 linhas duplicadas na chave ['CNPJ_CIA', 'ANO']
   - giro_ativo com valor extremo (4,010,405), revisar winsorizacao
   - kanitz_fi com valor extremo (2,563,220), revisar winsorizacao
   - altman_z2_emergentes com valor extremo (2,402,848), revisar winsorizacao

A CVM MG (773 linhas)
   - apenas 3 positivos, insuficiente para K-Fold estratificado com k=10
   - giro_ativo com valor extremo (4,010,405), revisar winsorizacao

B BACEN nacional (16,552 linhas)
   - nenhum caso positivo
   - roa com 56% de ausentes
   - roe com 56% de ausentes
   - pl_sobre_ativo com valor extremo (2,446,464), revisar winsorizacao

B BACEN MG (2,478 linhas)
   

In [48]:
if "y_rj" in camada_a.columns and camada_a["y_rj"].sum() > 0:
    base = camada_a[camada_a["descartar_pos_evento"] == 0]
    medianas = base.groupby("y_rj")[ind_a].median().T
    medianas.columns = ["saudaveis", "pre_insolvencia"]
    medianas["variacao"] = (medianas["pre_insolvencia"] - medianas["saudaveis"]) / medianas["saudaveis"].abs()
    print("Medianas dos indicadores, camada A\n")
    print(medianas.sort_values("variacao").to_string())
else:
    print("Sem casos positivos em y_rj, comparacao de medianas nao aplicavel.")

Medianas dos indicadores, camada A

                      saudaveis  pre_insolvencia  variacao
roa                      0.0204          -0.1773   -9.7036
cobertura_juros          1.1738          -0.4376   -1.3728
altman_z2_emergentes     4.9396          -0.1300   -1.0263
liquidez_corrente        1.4060           0.6501   -0.5376
roe                      0.0895           0.0426   -0.5240
kanitz_fi                3.5965           2.0057   -0.4423
giro_ativo               0.3617           0.2514   -0.3048
endividamento_geral      0.6381           0.9409    0.4747


In [49]:
manifesto = {
    "gerado_em": datetime.now().isoformat(timespec="seconds"),
    "uf_recorte": UF_RECORTE,
    "anos_coorte_rfb": list(ANOS_COORTE),
    "janela_sobrevivencia_anos": JANELA_ANOS,
    "camada_a": {"observacoes": int(len(camada_a)),
                 "companhias": int(camada_a["CNPJ_CIA"].nunique()),
                 "anos": [int(camada_a["ANO"].min()), int(camada_a["ANO"].max())],
                 "eventos_aplicados": int(len(usados)),
                 "positivos_y_rj": int(camada_a["y_rj"].sum()),
                 "positivos_y_distress": int(camada_a["y_distress"].sum()),
                 "observacoes_mg": int(len(camada_a_mg)) if camada_a_mg is not None else 0},
    "camada_b": {"observacoes": int(len(camada_b)),
                 "instituicoes": int(camada_b["cod_inst"].nunique()) if len(camada_b) else 0,
                 "anos": sorted(int(a) for a in camada_b["ano"].unique()) if len(camada_b) else [],
                 "positivos": int(camada_b["y_resolucao"].sum()) if len(camada_b) else 0,
                 "observacoes_mg": int(len(camada_b_mg)) if camada_b_mg is not None else 0},
    "camada_c": {"empresas": int(len(camada_c)),
                 "positivos": int(camada_c["y_baixa"].sum()),
                 "taxa_baixa": float(camada_c["y_baixa"].mean())},
    "arquivos": sorted(p.name for p in DIR_FINAL.glob("*.parquet")),
}

(DIR_FINAL / "manifesto_tratamento.json").write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(manifesto, ensure_ascii=False, indent=2))
print("\nFIM DA ENTREGA 2")

{
  "gerado_em": "2026-09-09T22:22:50",
  "uf_recorte": "MG",
  "anos_coorte_rfb": [
    2015,
    2016,
    2017,
    2018,
    2019
  ],
  "janela_sobrevivencia_anos": 5,
  "camada_a": {
    "observacoes": 10172,
    "companhias": 1181,
    "anos": [
      2011,
      2026
    ],
    "eventos_aplicados": 48,
    "positivos_y_rj": 48,
    "positivos_y_distress": 2386,
    "observacoes_mg": 773
  },
  "camada_b": {
    "observacoes": 16552,
    "instituicoes": 1967,
    "anos": [
      2014,
      2015,
      2016,
      2017,
      2018,
      2019,
      2020,
      2021,
      2022,
      2023,
      2024,
      2025
    ],
    "positivos": 0,
    "observacoes_mg": 2478
  },
  "camada_c": {
    "empresas": 1415279,
    "positivos": 535270,
    "taxa_baixa": 0.37820811302930374
  },
  "arquivos": [
    "camada_a_cvm_mg.parquet",
    "camada_a_cvm_nacional.parquet",
    "camada_a_cvm_painel.parquet",
    "camada_a_cvm_pareado_distress.parquet",
    "camada_a_cvm_pareado_rj.parquet",
 

---
## O que fica pendente para a próxima etapa

**Preencher `02_intermediario/eventos_cvm_a_pesquisar.csv`.** São as companhias cujo carimbo de recuperação judicial está na razão social mas cuja data não pode ser derivada do cadastro. Cada linha preenchida vira um caso positivo a mais na camada A, que é a camada mais escassa em eventos.

**Conferir `02_intermediario/pareamento_resolucao.csv`.** O pareamento entre a lista de liquidações do BACEN e o cadastro do IF.data é feito por similaridade de nome. Marque `S` na coluna `confirmado` apenas nas linhas corretas.

**Conferir a lista de indicadores localizados na camada B.** A célula 3.2 imprime qual coluna do IF.data foi usada para cada indicador. Se algum aparecer como não encontrado, o nome mudou de redação e precisa ser acrescentado ao `MAPA_INDICADORES`.

A modelagem, com os três algoritmos, validação temporal e SHAP, entra na entrega 3.